In [1]:
import warnings
warnings.filterwarnings( 'ignore' )
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split


In [2]:
data_cleaned = pd.read_csv('../cleaned_gtd.csv', encoding='ISO-8859-1')
data_cleaned['attack_date'] = pd.to_datetime({'year': data_cleaned['iyear'], 'month': data_cleaned['imonth'], 'day': data_cleaned['iday']})


In [3]:
# Step 1: Get the top 30 class labels
top_30_classes = data_cleaned['gname'].value_counts().head(30).index

# Step 2: Filter the DataFrame to only include rows with those class labels
top_30_df = data_cleaned[data_cleaned['gname'].isin(top_30_classes)]

sample_size = 300
top_30_df = top_30_df.groupby('gname').sample(n=sample_size, random_state=42)
print(top_30_df['gname'].value_counts())


gname
Abu Sayyaf Group (ASG)                              300
African National Congress (South Africa)            300
Tehrik-i-Taliban Pakistan (TTP)                     300
Taliban                                             300
Sikh Extremists                                     300
Shining Path (SL)                                   300
Revolutionary Armed Forces of Colombia (FARC)       300
Palestinians                                        300
Nicaraguan Democratic Force (FDN)                   300
New People's Army (NPA)                             300
National Liberation Army of Colombia (ELN)          300
Muslim extremists                                   300
Maoists                                             300
Manuel Rodriguez Patriotic Front (FPMR)             300
Liberation Tigers of Tamil Eelam (LTTE)             300
Kurdistan Workers' Party (PKK)                      300
Islamic State of Iraq and the Levant (ISIL)         300
Irish Republican Army (IRA)               

In [4]:
train = top_30_df.drop(columns=['gname'])
for col in train.select_dtypes(include='object').columns:
    train[col], _ = pd.factorize(train[col])

top_30_df = pd.concat([train, top_30_df['gname']], axis=1)


In [5]:
len(top_30_df['gname'].value_counts())

30

In [6]:
top_30_df.tail()

,iyear,imonth,iday,extended,country,region,provstate,city,latitude,longitude,...,targtype1,target1,natlty1,individual,weaptype1,nkill,property,ishostkid,attack_date,gname
32223,1994,1,6,0,159,3,277,2111,-11.967368,-76.978462,...,1,4915,159.0,0,6,0.0,1,0.0,1994-01-06,Tupac Amaru Revolutionary Movement (MRTA)
19554,1988,5,12,0,159,3,277,2111,-11.967368,-76.978462,...,8,4916,159.0,0,5,0.0,1,0.0,1988-05-12,Tupac Amaru Revolutionary Movement (MRTA)
24251,1990,2,14,0,159,3,277,2111,-11.967368,-76.978462,...,1,283,217.0,0,6,0.0,1,0.0,1990-02-14,Tupac Amaru Revolutionary Movement (MRTA)
13620,1985,4,13,0,159,3,277,2111,-11.967368,-76.978462,...,11,4898,159.0,0,6,1.0,1,0.0,1985-04-13,Tupac Amaru Revolutionary Movement (MRTA)
28926,1991,11,4,0,159,3,277,2111,-11.967368,-76.978462,...,1,4917,159.0,0,6,0.0,1,0.0,1991-11-04,Tupac Amaru Revolutionary Movement (MRTA)


In [7]:
top_30_df.to_csv(f'df_top30_{sample_size}.csv')